In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType, TimestampType

BOOTSTRAP  = "kafka:9092"
TOPIC      = "taxi-trips"
CHECKPOINT = "/home/jovyan/checkpoints/bronze"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

bronze_stream = raw_stream.select(
    raw_stream.timestamp.alias("kafka_time"),
    raw_stream.value.cast("string").alias("value"),
    raw_stream.partition.alias("kafka_partition"),
    F.col("offset").alias("kafka_offset"),
)

bronze_query = (
    bronze_stream.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .toTable("lakehouse.taxi.bronze")
)

In [3]:
import time
time.sleep(10)
print("Bronze row count:", spark.read.table("lakehouse.taxi.bronze").count())
spark.read.table("lakehouse.taxi.bronze").printSchema()
spark.read.table("lakehouse.taxi.bronze").show(5, truncate=80)

Bronze row count: 1641
root
 |-- kafka_time: timestamp (nullable = true)
 |-- value: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)

+-----------------------+--------------------------------------------------------------------------------+---------------+------------+
|             kafka_time|                                                                           value|kafka_partition|kafka_offset|
+-----------------------+--------------------------------------------------------------------------------+---------------+------------+
|2026-04-05 14:40:11.521|{"VendorID": 1, "tpep_pickup_datetime": "2025-01-01T00:12:11", "tpep_dropoff_...|              0|         272|
|2026-04-05 14:40:11.532|{"VendorID": 1, "tpep_pickup_datetime": "2025-01-01T00:29:38", "tpep_dropoff_...|              0|         273|
|2026-04-05 14:40:11.542|{"VendorID": 1, "tpep_pickup_datetime": "2025-01-01T00:33:01", "tpep_dropoff_...|             

In [4]:
from pyspark.sql.functions import from_json, col, unix_timestamp, round

payload_schema = StructType([
    StructField("tpep_pickup_datetime", StringType()),
    StructField("fare_amount",          DoubleType()),
    StructField("PULocationID",         IntegerType()),
])

spark.read.table("lakehouse.taxi.bronze") \
    .withColumn("payload", from_json(col("value"), payload_schema)) \
    .withColumn("pickup_time", col("payload.tpep_pickup_datetime").cast("timestamp")) \
    .select(
        col("kafka_time"),
        col("pickup_time"),
        round((unix_timestamp("kafka_time") - unix_timestamp("pickup_time")) / 86400, 1).alias("diff_days")
    ).show(10, truncate=False)

+-----------------------+-------------------+---------+
|kafka_time             |pickup_time        |diff_days|
+-----------------------+-------------------+---------+
|2026-04-05 14:39:58.435|2025-01-01 00:18:38|459.6    |
|2026-04-05 14:39:58.545|2025-01-01 00:32:40|459.6    |
|2026-04-05 14:39:58.556|2025-01-01 00:44:04|459.6    |
|2026-04-05 14:39:58.606|2025-01-01 00:14:47|459.6    |
|2026-04-05 14:39:58.62 |2025-01-01 00:39:27|459.6    |
|2026-04-05 14:39:58.631|2025-01-01 00:53:43|459.6    |
|2026-04-05 14:39:58.686|2025-01-01 00:30:07|459.6    |
|2026-04-05 14:39:58.696|2025-01-01 00:39:55|459.6    |
|2026-04-05 14:39:58.707|2025-01-01 00:16:54|459.6    |
|2026-04-05 14:39:58.721|2025-01-01 00:43:10|459.6    |
+-----------------------+-------------------+---------+
only showing top 10 rows


In [5]:
count_before = spark.read.table("lakehouse.taxi.bronze").count()
print(f"Before restart: {count_before}")

bronze_query.stop()

bronze_query = (
    bronze_stream.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .toTable("lakehouse.taxi.bronze")
)

time.sleep(10)
count_after = spark.read.table("lakehouse.taxi.bronze").count()
print(f"After restart:  {count_after}")
print(f"Difference:     {count_after - count_before}")

Before restart: 1779
After restart:  1845
Difference:     66


In [6]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.taxi.silver (
        pickup_time          TIMESTAMP,
        dropoff_time         TIMESTAMP,
        vendor_id            INT,
        passenger_count      INT,
        trip_distance        DOUBLE,
        pu_location_id       INT,
        do_location_id       INT,
        fare_amount          DOUBLE,
        tip_amount           DOUBLE,
        total_amount         DOUBLE,
        payment_type         INT,
        congestion_surcharge DOUBLE,
        pu_zone              STRING,
        do_zone              STRING,
        kafka_time           TIMESTAMP
    )
    USING iceberg
""")
print("Silver table ready")

Silver table ready


In [7]:
silver_payload_schema = StructType([
    StructField("VendorID",              IntegerType()),
    StructField("tpep_pickup_datetime",  StringType()),
    StructField("tpep_dropoff_datetime", StringType()),
    StructField("passenger_count",       IntegerType()),
    StructField("trip_distance",         DoubleType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("fare_amount",           DoubleType()),
    StructField("tip_amount",            DoubleType()),
    StructField("total_amount",          DoubleType()),
    StructField("payment_type",          IntegerType()),
    StructField("congestion_surcharge",  DoubleType()),
])

zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
pu_zones = zones.select(F.col("LocationID").alias("pu_loc_id"), F.col("Zone").alias("pu_zone"))
do_zones = zones.select(F.col("LocationID").alias("do_loc_id"), F.col("Zone").alias("do_zone"))

silver_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

silver_stream = (
    silver_raw
    .withColumn("payload",     F.from_json(F.col("value").cast("string"), silver_payload_schema))
    .withColumn("pickup_time",  F.col("payload.tpep_pickup_datetime").cast("timestamp"))
    .withColumn("dropoff_time", F.col("payload.tpep_dropoff_datetime").cast("timestamp"))
    .filter(F.col("pickup_time").isNotNull())
    .filter(F.col("payload.fare_amount") > 0)
    .filter(F.col("payload.trip_distance") > 0)
    .join(F.broadcast(pu_zones), F.col("payload.PULocationID") == F.col("pu_loc_id"), "left")
    .join(F.broadcast(do_zones), F.col("payload.DOLocationID") == F.col("do_loc_id"), "left")
    .select(
        F.col("pickup_time"),
        F.col("dropoff_time"),
        F.col("payload.VendorID").alias("vendor_id"),
        F.col("payload.passenger_count").alias("passenger_count"),
        F.col("payload.trip_distance").alias("trip_distance"),
        F.col("payload.PULocationID").alias("pu_location_id"),
        F.col("payload.DOLocationID").alias("do_location_id"),
        F.col("payload.fare_amount").alias("fare_amount"),
        F.col("payload.tip_amount").alias("tip_amount"),
        F.col("payload.total_amount").alias("total_amount"),
        F.col("payload.payment_type").alias("payment_type"),
        F.col("payload.congestion_surcharge").alias("congestion_surcharge"),
        F.col("pu_zone"),
        F.col("do_zone"),
        F.col("timestamp").alias("kafka_time"),
    )
)

silver_query = (
    silver_stream.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/home/jovyan/checkpoints/silver")
    .toTable("lakehouse.taxi.silver")
)

In [8]:
time.sleep(15)
print("Silver row count:", spark.read.table("lakehouse.taxi.silver").count())
spark.read.table("lakehouse.taxi.silver").printSchema()
spark.read.table("lakehouse.taxi.silver").show(5, truncate=40)

Silver row count: 4106
root
 |-- pickup_time: timestamp (nullable = true)
 |-- dropoff_time: timestamp (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pu_location_id: integer (nullable = true)
 |-- do_location_id: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- pu_zone: string (nullable = true)
 |-- do_zone: string (nullable = true)
 |-- kafka_time: timestamp (nullable = true)

+-------------------+-------------------+---------+---------------+-------------+--------------+--------------+-----------+----------+------------+------------+--------------------+-----------------------------+-------------------+-----------------------+
|        pickup_time|       dropoff_time|ven

In [9]:
spark.sql("DROP TABLE IF EXISTS lakehouse.taxi.gold")

spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold (
        pickup_date        DATE,
        pickup_hour        TIMESTAMP,
        pu_zone            STRING,
        total_trips        LONG,
        total_revenue      DOUBLE,
        avg_fare           DOUBLE,
        avg_tip            DOUBLE,
        avg_trip_distance  DOUBLE
    )
    USING iceberg
    PARTITIONED BY (pickup_date)
""")
print("Gold table ready")

Gold table ready


In [10]:
import subprocess
subprocess.run(["rm", "-rf", "/home/jovyan/checkpoints/gold"])

gold_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .withColumn("payload",    F.from_json(F.col("value").cast("string"), silver_payload_schema))
    .withColumn("pickup_time", F.col("payload.tpep_pickup_datetime").cast("timestamp"))
    .filter(F.col("pickup_time").isNotNull())
    .filter(F.col("payload.fare_amount") > 0)
    .filter(F.col("payload.trip_distance") > 0)
    .join(F.broadcast(pu_zones), F.col("payload.PULocationID") == F.col("pu_loc_id"), "left")
    .withWatermark("pickup_time", "10 minutes")
    .groupBy(
        F.window("pickup_time", "1 hour").alias("window"),
        F.col("pu_zone")
    )
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("payload.fare_amount"), 2).alias("total_revenue"),
        F.round(F.avg("payload.fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("payload.tip_amount"),  2).alias("avg_tip"),
        F.round(F.avg("payload.trip_distance"), 2).alias("avg_trip_distance"),
    )
    .select(
        F.to_date(F.col("window.start")).alias("pickup_date"),
        F.col("window.start").alias("pickup_hour"),
        F.col("pu_zone"),
        F.col("total_trips"),
        F.col("total_revenue"),
        F.col("avg_fare"),
        F.col("avg_tip"),
        F.col("avg_trip_distance"),
    )
)

gold_query = (
    gold_stream.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/home/jovyan/checkpoints/gold")
    .toTable("lakehouse.taxi.gold")
)

In [11]:
time.sleep(15)
print("Gold row count:", spark.read.table("lakehouse.taxi.gold").count())
spark.read.table("lakehouse.taxi.gold").show(5, truncate=40)

Gold row count: 10
+-----------+-------------------+-------------------+-----------+-------------+--------+-------+-----------------+
|pickup_date|        pickup_hour|            pu_zone|total_trips|total_revenue|avg_fare|avg_tip|avg_trip_distance|
+-----------+-------------------+-------------------+-----------+-------------+--------+-------+-----------------+
| 2024-12-31|2024-12-31 23:00:00|  LaGuardia Airport|          1|         48.5|    48.5|  11.55|            12.89|
| 2024-12-31|2024-12-31 23:00:00|       Central Park|          1|         10.7|    10.7|    0.0|             1.03|
| 2024-12-31|2024-12-31 23:00:00|        Murray Hill|          1|         10.0|    10.0|    3.0|             1.53|
| 2024-12-31|2024-12-31 23:00:00|             Corona|          1|          7.9|     7.9|    0.0|             1.12|
| 2024-12-31|2024-12-31 23:00:00|Lincoln Square East|          1|         12.1|    12.1|   3.42|             2.44|
+-----------+-------------------+-------------------+--------

In [12]:
spark.sql("SELECT snapshot_id, committed_at, operation FROM lakehouse.taxi.gold.snapshots").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|4954389297534097177|2026-04-05 14:45:46.428|append   |
|2216769876623997446|2026-04-05 14:45:47.021|append   |
|7464892296813783624|2026-04-05 14:45:47.521|append   |
|5048547462411402582|2026-04-05 14:45:47.998|append   |
|8962554433342703174|2026-04-05 14:45:48.449|append   |
|5370211395919992290|2026-04-05 14:45:48.835|append   |
|4419729020071855207|2026-04-05 14:45:49.44 |append   |
|4018539708641143880|2026-04-05 14:45:50.029|append   |
|3774235479755807471|2026-04-05 14:45:50.494|append   |
|3091064122287054612|2026-04-05 14:45:50.977|append   |
|8663008664592660544|2026-04-05 14:45:51.331|append   |
|1789012320347081141|2026-04-05 14:45:51.702|append   |
|1919876444391895513|2026-04-05 14:45:52.229|append   |
|2649105699845505597|2026-04-05 14:45:52.625|append   |
|3650219419174150234|2026-04-05 14:45:53.074|app

In [31]:
print("Bronze:", spark.read.table("lakehouse.taxi.bronze").count())
print("Silver:", spark.read.table("lakehouse.taxi.silver").count())
print("Gold:  ", spark.read.table("lakehouse.taxi.gold").count())

Bronze: 1845
Silver: 11519
Gold:   207
